# Buril — pruebas y experimentos (Google Colab)

> **El entrenamiento de Buril ahora corre en Kaggle**
> (`tools/buril_kaggle.ipynb`): GPU T4 x2 gratis y *Save & Run All* en segundo
> plano, sin corte por inactividad. Este cuaderno de Colab queda para **probar
> otros modelos** (rivales, variantes, cuantizaciones) y experimentos cortos.
> Igual sigue funcionando para entrenar Buril si lo necesitás.

Cuaderno **driver fino**: no copia la lógica, sólo llama a los scripts del repo.
Así, cuando `tools/buril.py` o el dataset cambien, este cuaderno ya los usa en el
próximo `git pull`. La fuente de verdad son los `tools/*.py`.

Cadena: configurar versiones → clonar repo → **cache del `.npz`** en la release
`dataset-buril` (si no está, lo prepara y lo sube) → entrenar → publicar el
modelo en la release `modelo-buril` (el buzón que lee `android-apk.yml`).

**Versionado**: la celda 1 pide la versión del `.npz` y la del modelo (por
defecto ambas `v1.0.0-experimental.1a`). La versión del `.npz` va al nombre del
asset en `dataset-buril`; la del modelo va al manifiesto y a la release.

**Corte de Colab**: la celda 2 guarda los checkpoints por defecto en la
release `checkpoints-buril` (assets de GitHub), no en tu Drive. Mientras
entrena, la celda 5 los sube cada ~2 min; si la sesión se corta, la próxima
reanuda desde el último estado. También podés usar Google Drive como opción.
El tag lo elegís vos (`checkpoints-1.a`, etc.) y el nombre visible puede llevar
espacios (`NOMBRE_CHECKPOINTS`). Con los valores por defecto (30 épocas, lote
256, aumentación nativa de TF) el entrenamiento apunta a ~15 min.

**Antes de empezar**: guardá en *Colab Secrets* (icono de llave 🔑, barra
izquierda) el secreto `GH_TOKEN` con un PAT fine-grained de permiso
*Contents: Read and write* sobre este repo. No lo escribas en una celda.

> Si estás probando este cuaderno **antes de mergear el PR**, cambiá `RAMA` en la
> celda 1 por el nombre de tu rama (Colab resuelve `main` por defecto).



In [ ]:
# @title 1. Configuración, clonado y dependencias
REPO = "LuisAlejandro544/Dibujo-"           # @param {type:"string"}
RAMA = "main"                               # @param {type:"string"}
VERSION_DATOS  = "v1.0.0-experimental.1a"   # @param {type:"string"}
VERSION_MODELO = "v1.0.0-experimental.1a"   # @param {type:"string"}
TAG_DATOS  = "dataset-buril"                # @param {type:"string"}
TAG_MODELO = "modelo-buril"                 # @param {type:"string"}
# Tag = identificador de la release (sin espacios; guiones/puntos sí).
# Nombre = texto visible que vos elegís, p. ej. "Checkpoints Buril 1.a".
TAG_CHECKPOINTS = "checkpoints-buril"       # @param {type:"string"}
NOMBRE_CHECKPOINTS = "Checkpoints Buril v1.0.0-experimental.1a"  # @param {type:"string"}
LADO          = 512                         # @param {type:"integer"}
POR_CATEGORIA = 200                         # @param {type:"integer"}
EPOCAS        = 30                          # @param {type:"integer"}
AUMENTOS      = 2                           # @param {type:"integer"}
PARCHES       = 8                           # @param {type:"integer"}
BANDA         = 1                           # @param {type:"integer"}
LR            = 0.002                       # @param {type:"number"}
LOTE          = 256                          # @param {type:"integer"}
LOTE_FINAL    = 128                          # @param {type:"integer"}
FASE_RAPIDA   = 0.75                        # @param {type:"number"}
CALENTAMIENTO = 1                           # @param {type:"integer"}

import os, sys, subprocess

RUTA = "/content/Dibujo-"
if os.path.isdir(os.path.join(RUTA, ".git")):
    subprocess.run(["git", "fetch", "origin", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "checkout", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", RAMA], cwd=RUTA, check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA,
         f"https://github.com/{REPO}.git", RUTA], check=True)

os.chdir(RUTA)

# Usamos SIEMPRE el mismo intérprete del kernel (sys.executable) para que las
# dependencias instaladas y los scripts corran en el mismo entorno.
try:
    import tensorflow as tf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "tensorflow"], check=True)
    import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Repositorio listo en", RUTA, "| rama", RAMA)
print("TensorFlow", tf.__version__, "| GPU:", [g.name for g in gpus] or "ninguna (CPU)")



In [ ]:
# @title 2. Puntos de control: release de GitHub (Drive opcional)
# Colab cierra la sesion a los ~30 min o si se cae el internet. Por defecto
# los checkpoints viven en la release TAG_CHECKPOINTS: no llenan tu Drive y
# se comparten con Kaggle. Activa USAR_DRIVE solo si querés una copia local.
USAR_RELEASE = True  # @param {type:"boolean"}
USAR_DRIVE   = False # @param {type:"boolean"}
REINICIAR    = False # @param {type:"boolean"} (True = entrenar de cero)

import os, shutil

CKPT = ".cache/modelos/checkpoints"
if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CKPT = f"/content/drive/MyDrive/Dibujo/checkpoints_{VERSION_MODELO}"
    except Exception as exc:
        print("Sin Google Drive; uso checkpoints locales:", exc)

if REINICIAR and os.path.isdir(CKPT):
    shutil.rmtree(CKPT)
os.makedirs(CKPT, exist_ok=True)
print("Checkpoints locales en", CKPT)
print("Release de checkpoints:", TAG_CHECKPOINTS, "->", NOMBRE_CHECKPOINTS)



In [ ]:
# @title 3. Utilidades de GitHub (release como buzón)
import json, os, requests
from google.colab import userdata

TOKEN = userdata.get("GH_TOKEN")
if not TOKEN:
    raise SystemExit("Falta el secreto GH_TOKEN en Colab (permiso Contents: Read and write).")

CABECERAS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
API = f"https://api.github.com/repos/{REPO}"


def asegurar_release(tag, nombre, cuerpo):
    """Devuelve la release [tag] creándola si no existe."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code == 200:
        return r.json()
    r = requests.post(f"{API}/releases", headers=CABECERAS,
                      json={"tag_name": tag, "name": nombre, "body": cuerpo})
    r.raise_for_status()
    return r.json()


def bajar_asset(tag, nombre, destino):
    """Baja el asset [nombre] de la release [tag] a [destino]; True si existía."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code != 200:
        return False
    for a in r.json().get("assets", []):
        if a["name"] == nombre:
            datos = requests.get(
                a["url"], headers={**CABECERAS, "Accept": "application/octet-stream"})
            datos.raise_for_status()
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            with open(destino, "wb") as f:
                f.write(datos.content)
            print(f"Bajado {nombre} ({len(datos.content)} bytes) de {tag}")
            return True
    return False


def subir_asset(tag, ruta, nombre, nombre_release=None, cuerpo=None):
    """Sube (o reemplaza) [ruta] como asset [nombre] de la release [tag]."""
    release = asegurar_release(
        tag, nombre_release or tag, cuerpo or f"Buzón `{tag}`.")
    for viejo in requests.get(f"{API}/releases/{release['id']}/assets",
                              headers=CABECERAS).json():
        if viejo["name"] == nombre:
            requests.delete(f"{API}/releases/assets/{viejo['id']}", headers=CABECERAS)
    with open(ruta, "rb") as f:
        datos = f.read()
    subida = requests.post(
        f"https://uploads.github.com/repos/{REPO}/releases/{release['id']}/assets",
        headers={**CABECERAS, "Content-Type": "application/octet-stream"},
        params={"name": nombre}, data=datos)
    subida.raise_for_status()
    print(f"Publicado {nombre} ({len(datos)} bytes) en la release {tag}")


def actualizar_release(tag, nombre, cuerpo):
    """Fija el nombre y las notas de la release [tag]."""
    release = asegurar_release(tag, nombre, cuerpo)
    r = requests.patch(f"{API}/releases/{release['id']}", headers=CABECERAS,
                       json={"name": nombre, "body": cuerpo})
    r.raise_for_status()
    print(f"Release {tag} actualizada: {nombre}")


In [ ]:
# @title 4. Dataset .npz: recuperar de la release o prepararlo
DIR = ".cache/limpiar_trazos_planb/procesado"
NPZ = f"{DIR}/pares_esq_lado{LADO}.npz"
MANIF = f"{DIR}/manifiesto_esq_lado{LADO}.json"
ASSET_NPZ = f"pares_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.npz"
ASSET_MANIF = f"manifiesto_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.json"

asegurar_release(TAG_DATOS, "Dataset Buril (cache del .npz)",
                 "Cache del dataset comercial (Quick, Draw!, CC BY 4.0) que usa "
                 "`tools/buril.py`. Lo llena `tools/buril_colab.ipynb` para no "
                 "volver a bajar/preprocesar en cada sesión.")

if bajar_asset(TAG_DATOS, ASSET_NPZ, NPZ):
    bajar_asset(TAG_DATOS, ASSET_MANIF, MANIF)
    print(f"Dataset recuperado de la release {TAG_DATOS} ({VERSION_DATOS}).")
else:
    print("No hay cache para esta versión; preparando el dataset (puede tardar)...")
    subprocess.run([sys.executable,
                    "tools/preparar_dataset_limpiar_trazos_planb.py", "todo",
                    "--lado", str(LADO), "--por-categoria", str(POR_CATEGORIA),
                    "--version", VERSION_DATOS], check=True)
    subir_asset(TAG_DATOS, NPZ, ASSET_NPZ)
    if os.path.exists(MANIF):
        subir_asset(TAG_DATOS, MANIF, ASSET_MANIF)

print("npz listo:", NPZ, os.path.getsize(NPZ), "bytes")


In [ ]:
# @title 5. Entrenar Buril (reanuda desde la release de checkpoints)
from os.path import basename, splitext

SALIDA = ".cache/modelos/limpiar_trazos.tflite"
CUERPO_CHECKPOINTS = (
    f"Ultimo estado de entrenamiento de Buril {VERSION_MODELO} "
    "(pesos + optimizador) para reanudar desde Colab o Kaggle."
)

# 1) Reanudar: baja el ultimo estado de la release salvo que ya haya uno
#    local (para no pisar un avance mas nuevo de la misma sesion).
if USAR_RELEASE and not REINICIAR and not os.path.exists(os.path.join(CKPT, "estado.json")):
    if bajar_asset(TAG_CHECKPOINTS, "estado.json", os.path.join(CKPT, "estado.json")):
        print("Reanudando desde", TAG_CHECKPOINTS)
        for nombre in ["ultimo.weights.h5", "mejor.weights.h5", "optimizador.npz"]:
            bajar_asset(TAG_CHECKPOINTS, nombre, os.path.join(CKPT, nombre))
    else:
        print("No hay checkpoints en", TAG_CHECKPOINTS, "; se entrena de cero.")

# 2) Subida periodica (~2 min) para sobrevivir a un corte de Colab.
import threading, shutil

parar = threading.Event()


def _subir_periodico(segundos=120):
    temporal = "/content/.ckpt_subida"
    os.makedirs(temporal, exist_ok=True)
    while not parar.wait(segundos):
        for nombre in ["estado.json", "ultimo.weights.h5", "mejor.weights.h5",
                       "optimizador.npz"]:
            ruta = os.path.join(CKPT, nombre)
            if not os.path.exists(ruta):
                continue
            copia = os.path.join(temporal, nombre)
            try:
                shutil.copy2(ruta, copia)
                subir_asset(TAG_CHECKPOINTS, copia, nombre,
                            NOMBRE_CHECKPOINTS, CUERPO_CHECKPOINTS)
            except Exception as exc:  # noqa: BLE001
                print("checkpoint intermedio no se pudo subir:", exc)


hilo = None
if USAR_RELEASE:
    hilo = threading.Thread(target=_subir_periodico, daemon=True)
    hilo.start()
else:
    print("USAR_RELEASE=False: no se suben checkpoints a GitHub.")

# 3) Entrenar (la logica vive en tools/buril.py).
try:
    subprocess.run([sys.executable, "tools/buril.py",
                    "--pares", NPZ,
                    "--salida", SALIDA,
                    "--epocas", str(EPOCAS),
                    "--aumentos", str(AUMENTOS),
                    "--banda", str(BANDA),
                    "--lr", str(LR),
                    "--lote", str(LOTE),
                    "--lote-final", str(LOTE_FINAL),
                    "--fase-rapida", str(FASE_RAPIDA),
                    "--calentamiento", str(CALENTAMIENTO),
                    "--parches-por-imagen", str(PARCHES),
                    "--validacion-cada", "2",
                    "--cldice-iteraciones", "5",
                    "--pasos-por-ejecucion", "2",
                    "--distribucion", "auto",
                    "--checkpoints", CKPT,
                    "--reanudar", CKPT,
                    "--nombre", "Buril",
                    "--version", VERSION_MODELO], check=True)
finally:
    parar.set()
    if hilo is not None:
        hilo.join(timeout=60)

print("Modelo entrenado:", SALIDA, os.path.getsize(SALIDA), "bytes")



In [ ]:
# @title 6. Publicar el modelo y los checkpoints en las releases
CUERPO = "\n\n".join([
    f"Modelo **Buril {VERSION_MODELO}** (`limpiar_trazos.tflite`, limpiador de "
    "trazos float16) entrenado en Google Colab con `tools/buril_colab.ipynb`.",
    f"- Dataset: `{ASSET_NPZ}` (release `{TAG_DATOS}`).",
    f"- Épocas: {EPOCAS} · aumentos: {AUMENTOS} · parches/hoja: {PARCHES}.",
    f"- Schedule: lote {LOTE} (rápida) -> {LOTE_FINAL} (pulido), calentamiento {CALENTAMIENTO}.",
    f"- Objetivo: banda de {2 * BANDA + 1} px · LR {LR} (calentamiento + coseno) · pérdida BCE + Dice + clDice.",
    "- `android-apk.yml` baja este asset y lo inyecta en el APK.",
])
actualizar_release(TAG_MODELO, f"Modelo Buril {VERSION_MODELO}", CUERPO)

for archivo in [SALIDA, splitext(SALIDA)[0] + ".json"]:
    if os.path.exists(archivo):
        subir_asset(TAG_MODELO, archivo, basename(archivo))

# 7. Checkpoints (pesos + optimizador) en la release para reanudar.
if USAR_RELEASE:
    actualizar_release(TAG_CHECKPOINTS, NOMBRE_CHECKPOINTS, CUERPO_CHECKPOINTS)
    for nombre in ["estado.json", "ultimo.weights.h5", "mejor.weights.h5",
                   "optimizador.npz"]:
        ruta = os.path.join(CKPT, nombre)
        if os.path.exists(ruta):
            subir_asset(TAG_CHECKPOINTS, ruta, nombre,
                        NOMBRE_CHECKPOINTS, CUERPO_CHECKPOINTS)
else:
    print("USAR_RELEASE=False: checkpoints solo locales", CKPT)

with open(splitext(SALIDA)[0] + ".json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## Siguiente paso

La release `modelo-buril` ya tiene el `.tflite` de la versión pedida y el `.npz`
quedó cacheado en `dataset-buril`. Los checkpoints (pesos + optimizador) quedan en
la release `checkpoints-buril` para reanudar desde Colab o Kaggle sin pasar por
Drive. El workflow `android-apk.yml` baja el `.tflite` y lo inyecta en
`app/src/main/assets/modelos/` antes de `assembleDebug`; si encuentra la release
de checkpoints, también la descarga y la publica como artefacto de la corrida
(los checkpoints no entran al APK). Para armar el APK, corré *Actions → "APK
Android (assembleDebug)" → Run workflow*. Si la release del modelo está vacía, el
APK se arma igual y la app cae al adelgazado algorítmico.
